# Task 9 — TimesNet với thiếu nhân tạo (MNAR)

**Mục tiêu:** lấy WAPE/WPE của TimesNet khi tạo thiếu nhân tạo, để đối chiếu với TimesNet ở **Bảng 2** của bài báo (bài báo: WAPE 27,62%, WPE 1,43%). Task 9 **không** xuất nhu cầu làm đầu vào dự báo.

**Cách làm:** chạy đúng mã của tác giả [frn-50k-baseline @ `8af694a`](https://github.com/Dingdong-Inc/frn-50k-baseline/tree/8af694a09a5a427d1d50648d348c022565a781f2) (`latent_demand_recovery/exp/app.py`, `--model TimesNet --missing_rate 0.3`, seed 1024 như CLI gốc). Mô hình, hàm tạo thiếu `mimic_missing`, `set_seed` và phần huấn luyện/dự đoán dùng nguyên mã tác giả. Vì bản gốc cần hơn 12,7 GB RAM (Colab miễn phí bị tắt tiến trình), hai hàm `load_data` và `evaluation_mnar` được **viết lại để tiết kiệm RAM nhưng giữ nguyên công thức và thứ tự gọi số ngẫu nhiên**: đọc `train.parquet` trên Drive thay vì Hugging Face, lưu đầu vào ở float32 (pypots vốn đổi sang float32 trước khi đưa vào mô hình), dùng chung mảng đã nạp. Lượt chạy thử **tự kiểm chứng** bản viết lại cho kết quả trùng bản gốc (dòng `equivalence_ok=True` trong log).

**Lưu ý về `missing_rate = 0.3`:** đây là **tham số mô phỏng theo ví dụ trong README của repo**, không khẳng định tỷ lệ thiếu thực tế đúng 30% hay là cấu hình chính xác của bài báo. Tỷ lệ che thực tế được in ra trong log (`actual_mask_ratio`).

**Trước khi chạy:** *Thời gian chạy → Thay đổi loại thời gian chạy → GPU (T4)*. Chạy lần lượt từng ô từ trên xuống. Ô chạy thử (mục 5) mất vài phút; ô chạy đầy đủ (mục 6) mất khoảng 2–3 giờ — giữ tab Colab mở, không để máy ngủ.

**Đầu ra trên Drive** (thư mục `OUT_DIR`, cấu trúc giống thư mục task trên GitHub):
- `results/mnar_metrics.csv` — 1 dòng TimesNet
- `results/full/timesnet_mnar.log`, `results/smoke/timesnet_mnar_smoke.log`
- `results/run_metadata.json`, `run_info.md`

## 0. Cấu hình

In [1]:
from pathlib import Path
import os, sys, json, platform, subprocess, re, time
from datetime import datetime, timezone

from google.colab import drive
drive.mount('/content/drive')

# ---- Sửa 2 dòng này nếu dữ liệu / thư mục kết quả của bạn ở chỗ khác ----
DATA_PATH = Path('/content/drive/MyDrive/FRN50K/raw/data/train.parquet')   # train.parquet (bản Task 5 tải từ HF, hoặc bản Task 6)
OUT_DIR   = Path('/content/drive/MyDrive/PTDLKD/task09_outputs')            # nơi lưu kết quả bàn giao

# ---- Tham số chạy (giữ nguyên trừ khi nhóm chốt khác) ----
MODEL        = 'TimesNet'
MISSING_RATE = 0.3      # tham số mô phỏng theo ví dụ README của repo tác giả
SEED         = 1024     # seed của CLI gốc app.py
EPOCHS       = 5        # mặc định CONFIG của tác giả
BATCH_SIZE   = 128      # mặc định CONFIG của tác giả
SMOKE_PAIRS  = 200      # chạy thử: 200 cặp đầu (18.000 dòng)
SMOKE_EPOCHS = 5        # chạy thử: 5 epoch (patience của tác giả = 5, không được lớn hơn số epoch)

AUTHOR_REPO   = 'https://github.com/Dingdong-Inc/frn-50k-baseline.git'
AUTHOR_COMMIT = '8af694a09a5a427d1d50648d348c022565a781f2'
AUTHOR_DIR    = Path('/content/frn-50k-baseline')
AUTHOR_EXP    = AUTHOR_DIR / 'latent_demand_recovery' / 'exp'
RUNNER        = Path('/content/run_task09.py')
WORK_ROOT     = Path('/content/task09_work')   # imputation .npy để local, không lên Drive
CKPT_DIR      = OUT_DIR / 'checkpoints'         # checkpoint lượt đầy đủ lưu trên Drive (KHÔNG đưa lên GitHub)
TASK6_TRAIN_SHA256 = 'b7e01c711bc4fa0598c4a8cf4e8b97769275b1afa1792d98fd8428c76838bc96'
PAPER = {'wape_pct': 27.62, 'wpe_pct': 1.43}   # Bảng 2, cột TimesNet (chỉ để đối chiếu)

for sub in ['results/full', 'results/smoke', 'checkpoints']:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
if not DATA_PATH.is_file():
    # Drive chưa có dữ liệu (ví dụ người khác chạy giúp): tải đúng file train.parquet gốc từ Hugging Face
    from huggingface_hub import hf_hub_download
    print('Chưa có train.parquet trên Drive -> đang tải từ Hugging Face (khoảng 100 MB)...')
    hf_hub_download(repo_id='Dingdong-Inc/FreshRetailNet-50K', repo_type='dataset',
                    filename='data/train.parquet', local_dir=str(DATA_PATH.parent.parent))
print('Dữ liệu:', DATA_PATH, '| tồn tại:', DATA_PATH.is_file())
print('Kết quả:', OUT_DIR)

Mounted at /content/drive
Dữ liệu: /content/drive/MyDrive/FRN50K/raw/data/train.parquet | tồn tại: True
Kết quả: /content/drive/MyDrive/PTDLKD/task09_outputs


## 1. Kiểm tra GPU và RAM

In [2]:
import psutil
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'KHÔNG CÓ GPU')
ram = psutil.virtual_memory().total / 1024**3
print(f'RAM hệ thống: {ram:.1f} GB')
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('>>> Chưa bật GPU: Thời gian chạy → Thay đổi loại thời gian chạy → chọn GPU, rồi chạy lại từ đầu.')
if ram < 12:
    print('>>> RAM dưới 12 GB: lượt chạy đầy đủ có thể bị hết RAM. Nếu có, chọn runtime "RAM cao".')

Wed Nov 18 01:35:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Tải mã tác giả và cài thư viện
Dùng PyTorch/NumPy có sẵn của Colab (không cài lại torch 2.0.1 vì không có bản cho Python mới của Colab). Chỉ cài thêm các gói mà `pypots` đi kèm repo cần, với `--no-deps` để không đụng tới torch.

In [3]:
if not AUTHOR_DIR.exists():
    subprocess.run(['git', 'clone', '-q', AUTHOR_REPO, str(AUTHOR_DIR)], check=True)
subprocess.run(['git', '-C', str(AUTHOR_DIR), 'checkout', '-q', AUTHOR_COMMIT], check=True)
head = subprocess.run(['git', '-C', str(AUTHOR_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Commit mã tác giả:', head, '| đúng 8af694a:', head == AUTHOR_COMMIT)

def pip(*pkgs):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *pkgs], capture_output=True, text=True)
    print('pip install', *pkgs, '->', 'OK' if r.returncode == 0 else r.stderr[-500:])

pip('pygrinder==0.4', 'tsdb==0.3.1')
for mod, pkg in [('einops', 'einops'), ('h5py', 'h5py'), ('sklearn', 'scikit-learn'), ('sympy', 'sympy')]:
    try:
        __import__(mod)
    except ImportError:
        pip(pkg)

Commit mã tác giả: 8af694a09a5a427d1d50648d348c022565a781f2 | đúng 8af694a: True
pip install pygrinder==0.4 tsdb==0.3.1 -> OK


In [4]:
# Kiểm tra import pypots đi kèm repo (chạy trong tiến trình riêng, giống lúc chạy thật)
check = f'''
import sys
sys.path[:0] = [{str(AUTHOR_EXP)!r}, {str(AUTHOR_EXP.parent)!r}]
import pypots, pygrinder, tsdb, torch, numpy, pandas
from pypots.imputation import TimesNet
print("pypots:", pypots.__file__)
print("pygrinder", pygrinder.__version__, "| tsdb", tsdb.__version__)
print("torch", torch.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)
'''
r = subprocess.run([sys.executable, '-c', check], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-3000:])
if r.returncode != 0 and 'pygrinder' in r.stderr:
    print('>>> pygrinder 0.4 lỗi với môi trường mới; thử bản mới nhất...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-U', 'pygrinder'])
    r = subprocess.run([sys.executable, '-c', check], capture_output=True, text=True)
    print(r.stdout); print(r.stderr[-3000:])
print('IMPORT OK' if r.returncode == 0 else '>>> IMPORT LỖI – chụp lỗi gửi lại để sửa trước khi chạy tiếp')

pypots: /content/frn-50k-baseline/latent_demand_recovery/pypots/__init__.py
pygrinder 0.4 | tsdb 0.3.1
torch 2.14.1+cu130 | numpy 2.4.6 | pandas 2.2.3

I0000 00:00:1794965731.122297    1202 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794965734.074665    1202 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

IMPORT OK


## 3. Xác nhận dữ liệu train

In [5]:
from hashlib import sha256
import pandas as pd, pyarrow.parquet as pq

def sha256_file(path):
    h = sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

DATA_SHA256 = sha256_file(DATA_PATH)
meta = pq.ParquetFile(DATA_PATH).metadata
keys = pd.read_parquet(DATA_PATH, columns=['store_id', 'product_id', 'dt'])
per_pair = keys.groupby(['store_id', 'product_id']).size()
DATA_INFO = {
    'path': str(DATA_PATH), 'sha256': DATA_SHA256, 'bytes': DATA_PATH.stat().st_size,
    'rows': int(meta.num_rows), 'columns': int(meta.num_columns),
    'pairs': int(len(per_pair)), 'all_pairs_90_days': bool((per_pair == 90).all()),
    'dt_min': str(keys['dt'].min()), 'dt_max': str(keys['dt'].max()),
    'duplicate_keys': int(keys.duplicated().sum()),
    'same_file_as_task6': DATA_SHA256 == TASK6_TRAIN_SHA256,
}
del keys
print(json.dumps(DATA_INFO, indent=2, ensure_ascii=False))
assert DATA_INFO['rows'] == 4_500_000 and DATA_INFO['pairs'] == 50_000 and DATA_INFO['all_pairs_90_days']
assert DATA_INFO['duplicate_keys'] == 0
print('Dữ liệu train hợp lệ: 4.500.000 dòng, 50.000 cặp × 90 ngày.',
      '(Trùng file Parquet của Task 6)' if DATA_INFO['same_file_as_task6'] else
      '(File tải trực tiếp từ Hugging Face, khác file Task 6 nhưng cùng nội dung train – ghi rõ trong run_info)')

{
  "path": "/content/drive/MyDrive/FRN50K/raw/data/train.parquet",
  "sha256": "6706832db892bbae4969c19d87e07975d2543d2ba7d7d4756360654785de5a3d",
  "bytes": 106436287,
  "rows": 4500000,
  "columns": 19,
  "pairs": 50000,
  "all_pairs_90_days": true,
  "dt_min": "2024-03-28",
  "dt_max": "2024-06-25",
  "duplicate_keys": 0,
  "same_file_as_task6": false
}
Dữ liệu train hợp lệ: 4.500.000 dòng, 50.000 cặp × 90 ngày. (File tải trực tiếp từ Hugging Face, khác file Task 6 nhưng cùng nội dung train – ghi rõ trong run_info)


## 4. Ghi script chạy
Script nhập `app.py` của tác giả, thay `load_data`/`evaluation_mnar` bằng bản tiết kiệm RAM (cùng công thức), in thêm các dòng `[task09]` (nguồn dữ liệu, số điểm che thêm, tỷ lệ che thực tế, RAM) rồi gọi đúng `set_seed` → `demand_recovery` như CLI gốc. Với `--verify_equivalence` (chỉ dùng ở lượt chạy thử), script chạy cả bản gốc lẫn bản viết lại trên cùng dữ liệu, cùng seed và dừng nếu kết quả khác nhau.

In [6]:
RUNNER.write_text('"""Task 9 runner: chạy TimesNet MNAR bằng mã gốc của tác giả (frn-50k-baseline @ 8af694a).\n\nMô hình, hàm tạo thiếu `mimic_missing`, `set_seed`, `_imputation` (fit/predict) dùng nguyên mã tác giả.\nHai hàm nạp dữ liệu/tính chỉ số của tác giả được viết lại để tiết kiệm RAM (Colab 12,7 GB không đủ\ncho bản gốc), giữ nguyên công thức và thứ tự gọi số ngẫu nhiên:\n  - load_data: đọc train.parquet local thay vì Hugging Face; X lưu float32 (pypots vốn đổi X sang\n    float32 trước khi đưa vào mô hình) và cấp phát một lần thay vì np.concatenate nhiều lần.\n  - evaluation_mnar: cùng công thức WAPE/WPE, dùng mảng đã nạp sẵn thay vì nạp lại dữ liệu.\nVới --verify_equivalence, script chạy cả bản gốc và bản viết lại trên cùng dữ liệu, cùng seed và\nxác nhận kết quả trùng khớp trước khi huấn luyện.\n"""\nimport argparse, contextlib, gc, io, json, os, sys, time\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow.compute as pc\nimport pyarrow.parquet as pq\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--author_exp\', required=True, help=\'.../frn-50k-baseline/latent_demand_recovery/exp\')\nap.add_argument(\'--data_path\', required=True, help=\'train.parquet\')\nap.add_argument(\'--work_dir\', required=True, help=\'thư mục chạy (checkpoint, imputation .npy)\')\nap.add_argument(\'--model\', default=\'TimesNet\')\nap.add_argument(\'--missing_rate\', type=float, default=0.3)\nap.add_argument(\'--seed\', type=int, default=1024)\nap.add_argument(\'--epochs\', type=int, default=5)\nap.add_argument(\'--batch_size\', type=int, default=128)\nap.add_argument(\'--pairs\', type=int, default=0, help=\'0 = toàn bộ; >0 = chỉ lấy N cặp đầu (chạy thử)\')\nap.add_argument(\'--saving_path\', default=None, help=\'thư mục lưu checkpoint (mặc định work_dir/save)\')\nap.add_argument(\'--resume_ckpt\', default=None,\n                help=\'file .pypots đã huấn luyện: bỏ qua fit, nạp checkpoint rồi predict + evaluation_mnar\')\nap.add_argument(\'--verify_equivalence\', action=\'store_true\',\n                help=\'so bản gốc và bản viết lại của load_data/evaluation_mnar (chỉ dùng cho tập nhỏ)\')\nargs = ap.parse_args()\nargs.work_dir = os.path.abspath(args.work_dir)\nargs.saving_path = os.path.abspath(args.saving_path) if args.saving_path else None\nargs.resume_ckpt = os.path.abspath(args.resume_ckpt) if args.resume_ckpt else None\n\n\ndef ram_gb():\n    try:\n        import psutil\n        return psutil.Process().memory_info().rss / 1024 ** 3\n    except Exception:\n        return float(\'nan\')\n\n\nSCALAR = [\'store_id\', \'product_id\', \'dt\', \'sale_amount\', \'discount\', \'holiday_flag\',\n          \'precpt\', \'avg_temperature\', \'stock_hour6_22_cnt\']\nLISTS = [\'hours_sale\', \'hours_stock_status\']\n\n\ndef read_train(path, n_pairs):\n    """Đọc train, sắp xếp theo (store_id, product_id, dt) như mã tác giả."""\n    df = pq.read_table(path, columns=SCALAR).to_pandas()\n    mats = {}\n    for col in LISTS:\n        arr = pq.read_table(path, columns=[col]).column(col).combine_chunks()\n        lens = pc.list_value_length(arr).to_numpy(zero_copy_only=False)\n        if not (lens == 24).all():\n            raise ValueError(f\'{col}: có dòng không đủ 24 giờ\')\n        mats[col] = arr.flatten().to_numpy(zero_copy_only=False).reshape(-1, 24)\n        del arr\n    order = np.lexsort((pd.to_datetime(df[\'dt\']).to_numpy(),\n                        df[\'product_id\'].to_numpy(), df[\'store_id\'].to_numpy()))\n    df = df.iloc[order].reset_index(drop=True)\n    for col in LISTS:\n        mats[col] = mats[col][order]\n    if n_pairs > 0:\n        key = df[\'store_id\'].astype(np.int64) * 1_000_000 + df[\'product_id\'].astype(np.int64)\n        keep = key.isin(pd.unique(key)[:n_pairs]).to_numpy()\n        df = df[keep].reset_index(drop=True)\n        for col in LISTS:\n            mats[col] = mats[col][keep]\n    size = df.groupby([\'store_id\', \'product_id\']).size()\n    if not (size == 90).all() or len(df) != len(size) * 90:\n        raise ValueError(\'Có cặp không đủ 90 ngày train\')\n    if not np.isin(mats[\'hours_stock_status\'], [0, 1]).all():\n        raise ValueError(\'hours_stock_status có giá trị khác 0/1\')\n    mats[\'hours_stock_status\'] = mats[\'hours_stock_status\'].astype(np.int8)   # chỉ so sánh == 1 và cộng\n    gc.collect()\n    return df, mats, int(len(size))\n\n\nt_start = time.time()\nTRAIN, MATS, N_PAIRS = read_train(args.data_path, args.pairs)\nSERIES_NUM = len(TRAIN) // 90\nSTOCK_POINTS = int(MATS[\'hours_stock_status\'][:, 6:22].sum())\nprint(f\'[task09] data_path={args.data_path}\', flush=True)\nprint(f\'[task09] train_rows={len(TRAIN)} pairs={N_PAIRS} \'\n      f\'dt_min={TRAIN["dt"].min()} dt_max={TRAIN["dt"].max()} \'\n      f\'stockout_points_6_21={STOCK_POINTS} ram_gb={ram_gb():.1f}\', flush=True)\n\nexp_dir = os.path.abspath(args.author_exp)\nsys.path[:0] = [exp_dir, os.path.dirname(exp_dir)]   # exp/ (data, model) và pypots đi kèm repo\nos.makedirs(args.work_dir, exist_ok=True)\nos.chdir(args.work_dir)                                # app.py ghi ./demand/ và ./save/ vào đây\n\nimport data.generate_data as gd      # noqa: E402\nimport app                           # noqa: E402\nimport pypots, torch                 # noqa: E402\nprint(f\'[task09] pypots_path={os.path.dirname(pypots.__file__)} torch={torch.__version__} \'\n      f\'cuda={torch.cuda.is_available()} \'\n      f\'gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}\', flush=True)\n\nORIG_LOAD_DATA = app.load_data\nORIG_EVAL_MNAR = app.evaluation_mnar\n\n\ndef load_data_lowmem(CONFIG):\n    """Cùng phép tính với generate_data.load_data của tác giả, ít tốn RAM hơn."""\n    hours_sale_full = MATS[\'hours_sale\']\n    hours_stock_status = MATS[\'hours_stock_status\']\n    hours_sale_origin = hours_sale_full.reshape(SERIES_NUM * 3, 30, 24)[..., 6:22]\n    hours_stock_status = hours_stock_status.reshape(SERIES_NUM * 3, 30, 24)[..., 6:22]\n    hours_sale = np.where(hours_stock_status == 1, np.nan, hours_sale_origin)\n    covariate = TRAIN[[\'discount\', \'holiday_flag\', \'precpt\', \'avg_temperature\']].values.reshape(SERIES_NUM * 3, 30, 4)\n    covariate = covariate / (covariate.max(axis=1, keepdims=True) + 0.1)\n\n    hours_sale, valid_idx = gd.mimic_missing(hours_sale, p=CONFIG[\'missing_rate\'],\n                                             max_missing_patch=7, min_missing_patch=3)\n\n    # Tác giả: concat [doanh số, 4 biến ngoại sinh, giờ/15] ở float64 rồi pypots đổi sang float32.\n    # Ở đây cấp phát thẳng mảng float32 và điền từng kênh -> cùng giá trị đầu vào mô hình.\n    X = np.empty(hours_sale.shape + (6,), dtype=np.float32)\n    X[..., 0] = hours_sale\n    X[..., 1:5] = covariate[:, :, None, :]\n    X[..., 5] = np.arange(16) / 15\n    del hours_sale\n    X = X.reshape(-1, 30 * 16, 6)\n    valid_idx = valid_idx.reshape(-1, 30 * 16, 1)\n    print(valid_idx.sum())\n    hours_sale_origin = hours_sale_origin.reshape(-1, 30 * 16, 1)\n    # pypots tự tạo norm_val = np.ones(X.shape) ở float64 (3,5 GB) rồi copy sang float32 (1,7 GB) mỗi lần\n    # tạo dataset -> hết RAM ngay khi bắt đầu predict. Truyền sẵn norm_val toàn số 1 dạng (n, 1, 6):\n    # mô hình nhân dec_out * norm_val theo broadcast nên kết quả y hệt (nhân với 1), chỉ tốn 3,6 MB.\n    norm_val = np.ones((X.shape[0], 1, X.shape[-1]), dtype=np.float32)\n    out = {\n        \'train_set\': {\'X\': X, \'norm_val\': norm_val},\n        \'ts_origin\': hours_sale_origin,\n        \'valid_idx\': valid_idx,\n        \'params\': {\'n_steps\': 30 * 16, \'n_features\': 6, \'patch_len\': 16, \'OT\': 1},\n    }\n    n_total = int(X[..., 0].size)\n    n_nan = int(np.isnan(X[..., 0]).sum())\n    n_art = int(valid_idx.sum())\n    n_stock = n_nan - n_art\n    print(f\'[task09] n_samples={X.shape[0]} n_points_6_21={n_total} \'\n          f\'stockout_points={n_stock} (khớp dữ liệu: {n_stock == STOCK_POINTS}) \'\n          f\'artificially_masked={n_art} actual_mask_ratio={n_art / (n_total - n_stock):.6f} \'\n          f\'ram_gb={ram_gb():.1f}\', flush=True)\n    gc.collect()\n    return out\n\n\ndef evaluation_mnar_lowmem(X, imputation):\n    """Cùng công thức với app.evaluation_mnar của tác giả, dùng mảng đã nạp sẵn."""\n    hours_sale_origin = MATS[\'hours_sale\'].reshape(SERIES_NUM * 3, 30, 24)\n    stock_hour_X = np.isnan(X[..., 0].reshape(-1, 30, 16)).sum(axis=-1).reshape(-1, 90).reshape(-1)\n    stock_hour_origin = MATS[\'hours_stock_status\'][:, 6:22].sum(axis=1)\n    hours_sale_impute = hours_sale_origin.copy()\n    hours_sale_impute[..., 6:22] = imputation.reshape(-1, 30, 16)\n    sale_amount_pred = hours_sale_impute.sum(axis=-1).reshape(-1, 90).reshape(-1)\n    valid_idx = (stock_hour_X > 0) & (stock_hour_origin == 0)\n    sale_amount = TRAIN[\'sale_amount\'].values\n    print(\'wape\', (np.abs(sale_amount_pred - sale_amount) * valid_idx).sum() / (sale_amount * valid_idx).sum())\n    print(\'wpe\', ((sale_amount_pred - sale_amount) * valid_idx).sum() / (sale_amount * valid_idx).sum())\n    print(f\'[task09] eval_days={int(valid_idx.sum())} ram_gb={ram_gb():.1f}\', flush=True)\n\n\ndef verify_equivalence(CONFIG):\n    """So bản gốc của tác giả với bản viết lại trên cùng dữ liệu, cùng seed."""\n    df_hf = TRAIN.copy()                     # giả lập dataset[\'train\'].to_pandas() của Hugging Face\n    for col in LISTS:\n        obj = np.empty(len(df_hf), dtype=object)\n        m = MATS[col].astype(np.int64) if col == \'hours_stock_status\' else MATS[col]\n        for i in range(len(df_hf)):\n            obj[i] = m[i]\n        df_hf[col] = obj\n\n    class _Split:\n        def to_pandas(self):\n            return df_hf.sample(frac=1, random_state=0)   # xáo trộn: tác giả tự sort lại\n\n    fake_hf = lambda *a, **k: {\'train\': _Split()}\n    gd.load_dataset = fake_hf\n    app.load_dataset = fake_hf\n    quiet = io.StringIO\n    with contextlib.redirect_stdout(quiet()):\n        app.set_seed(seed_value=args.seed)\n        ref = ORIG_LOAD_DATA(CONFIG)\n        app.set_seed(seed_value=args.seed)\n        new = load_data_lowmem(CONFIG)\n    same_X = np.array_equal(ref[\'train_set\'][\'X\'].astype(np.float32), new[\'train_set\'][\'X\'], equal_nan=True)\n    same_mask = np.array_equal(ref[\'valid_idx\'], new[\'valid_idx\'])\n    same_origin = np.array_equal(ref[\'ts_origin\'], new[\'ts_origin\'])\n    rng = np.random.default_rng(0)\n    imp = (rng.random(new[\'train_set\'][\'X\'].shape[:2] + (1,)) * 0.3).astype(np.float32)\n    buf_ref, buf_new = quiet(), quiet()\n    with contextlib.redirect_stdout(buf_ref):\n        ORIG_EVAL_MNAR(ref[\'train_set\'][\'X\'], imp)\n    with contextlib.redirect_stdout(buf_new):\n        evaluation_mnar_lowmem(new[\'train_set\'][\'X\'], imp)\n    lines_ref = [l for l in buf_ref.getvalue().splitlines() if l.startswith(\'wape\') or l.startswith(\'wpe\')]\n    lines_new = [l for l in buf_new.getvalue().splitlines() if l.startswith(\'wape\') or l.startswith(\'wpe\')]\n    same_eval = lines_ref == lines_new and len(lines_ref) == 2\n    ok = same_X and same_mask and same_origin and same_eval\n    print(f\'[task09] equivalence X={same_X} valid_idx={same_mask} ts_origin={same_origin} \'\n          f\'evaluation_mnar={same_eval} ({lines_ref} vs {lines_new}) equivalence_ok={ok}\', flush=True)\n    gd.load_dataset = app.load_dataset = None\n    del df_hf, ref, new\n    gc.collect()\n    if not ok:\n        raise SystemExit(\'Bản viết lại KHÔNG trùng bản gốc – dừng, không huấn luyện.\')\n\n\napp.load_data = load_data_lowmem\napp.evaluation_mnar = evaluation_mnar_lowmem\n\napp.CONFIG.update({\n    \'model\': args.model,\n    \'missing_rate\': args.missing_rate,\n    \'EPOCHS\': args.epochs,\n    \'batch_size\': args.batch_size,\n    \'saving_path\': args.saving_path or os.path.join(args.work_dir, \'save\'),\n})\nprint(\'[task09] CONFIG \' + json.dumps(app.CONFIG, ensure_ascii=False), flush=True)\nprint(f\'[task09] seed={args.seed}\', flush=True)\n\nif args.verify_equivalence:\n    verify_equivalence(app.CONFIG)\n\n_orig_load_model = app.load_model\n\n\ndef load_model_logged(CONFIG):\n    m = _orig_load_model(CONFIG)\n    _orig_fit, _orig_predict = m.fit, m.predict\n\n    def fit_logged(*a, **k):\n        print(f\'[task09] fit_start ram_gb={ram_gb():.1f}\', flush=True)\n        if args.resume_ckpt:\n            # Tiếp tục sau khi bị ngắt: giữ nguyên _imputation của tác giả, thay huấn luyện bằng nạp checkpoint.\n            # set_seed trước load_data như CLI gốc -> cùng mặt nạ thiếu như lượt đã huấn luyện.\n            loaded = torch.load(args.resume_ckpt, map_location=m.device, weights_only=False)\n            state = loaded.state_dict() if isinstance(loaded, torch.nn.Module) else loaded.model.state_dict()\n            m.model.load_state_dict(state)\n            print(f\'[task09] resumed_from_checkpoint={args.resume_ckpt} (bỏ qua huấn luyện)\', flush=True)\n        else:\n            _orig_fit(*a, **k)\n        print(f\'[task09] fit_end ram_gb={ram_gb():.1f}\', flush=True)\n\n    def predict_logged(*a, **k):\n        print(f\'[task09] predict_start ram_gb={ram_gb():.1f}\', flush=True)\n        res = _orig_predict(*a, **k)\n        print(f\'[task09] predict_end shape={res["imputation"].shape} ram_gb={ram_gb():.1f}\', flush=True)\n        return res\n\n    m.fit, m.predict = fit_logged, predict_logged\n    return m\n\n\napp.load_model = load_model_logged\n\napp.set_seed(seed_value=args.seed)   # như CLI gốc: set_seed rồi demand_recovery\napp.demand_recovery(app.CONFIG)      # missing_rate > 0 -> _imputation -> evaluation_mnar (in wape, wpe)\nprint(f\'[task09] finished elapsed_sec={time.time() - t_start:.0f} ram_gb={ram_gb():.1f}\', flush=True)\n', encoding='utf-8')
print('Đã ghi', RUNNER)

Đã ghi /content/run_task09.py


In [7]:
def run_logged(extra_args, log_path, work_dir):
    cmd = [sys.executable, '-u', str(RUNNER), '--author_exp', str(AUTHOR_EXP), '--data_path', str(DATA_PATH),
           '--work_dir', str(work_dir), '--model', MODEL, '--missing_rate', str(MISSING_RATE),
           '--seed', str(SEED), '--batch_size', str(BATCH_SIZE), *extra_args]
    print('Lệnh:', ' '.join(cmd), '\n')
    t0 = time.time()
    with open(log_path, 'w', encoding='utf-8') as log:
        log.write('# command: ' + ' '.join(cmd) + '\n')
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='')
            log.write(line); log.flush()
        p.wait()
    print(f'\n== mã thoát {p.returncode}, {time.time() - t0:.0f} giây. Log: {log_path}')
    return cmd, p.returncode

NUM = r'(-?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?|nan)'
def parse_log(log_path):
    text = Path(log_path).read_text(encoding='utf-8')
    def grab(pattern, cast=float):
        m = re.findall(pattern, text, flags=re.M)
        return cast(m[-1]) if m else None
    return {
        'wape': grab(r'^wape\s+' + NUM),
        'wpe': grab(r'^wpe\s+' + NUM),
        'artificially_masked': grab(r'artificially_masked=(\d+)', int),
        'actual_mask_ratio': grab(r'actual_mask_ratio=' + NUM),
        'stockout_points': grab(r'stockout_points=(\d+)', int),
        'n_samples': grab(r'n_samples=(\d+)', int),
        'train_rows': grab(r'train_rows=(\d+)', int),
        'elapsed_sec': grab(r'finished elapsed_sec=(\d+)', int),
        'equivalence_ok': (lambda v: None if v is None else v == 'True')(grab(r'equivalence_ok=(True|False)', str)),
        'eval_days': grab(r'eval_days=(\d+)', int),
        'peak_ram_gb': max([float(x) for x in re.findall(r'ram_gb=([\d.]+)', text)] or [float('nan')]),
    }

## 5. Chạy thử trên tập nhỏ
200 cặp đầu (theo thứ tự `store_id, product_id`), 5 epoch (không thể ít hơn vì `patience = 5` trong cấu hình tác giả). Mục đích: kiểm tra môi trường và luồng chạy, **không** dùng số này làm kết quả.

In [8]:
SMOKE_LOG = OUT_DIR / 'results/smoke/timesnet_mnar_smoke.log'
smoke_cmd, smoke_rc = run_logged(['--pairs', str(SMOKE_PAIRS), '--epochs', str(SMOKE_EPOCHS), '--verify_equivalence'],
                                 SMOKE_LOG, WORK_ROOT / 'smoke')
smoke = parse_log(SMOKE_LOG)
print(json.dumps(smoke, indent=2))
ok = smoke_rc == 0 and smoke['wape'] is not None and smoke['wpe'] is not None and smoke['equivalence_ok'] is True
print('\nCHẠY THỬ OK – chạy tiếp mục 6' if ok else '\n>>> CHẠY THỬ LỖI – chụp phần cuối log gửi lại, chưa chạy mục 6')

Lệnh: /usr/bin/python3 -u /content/run_task09.py --author_exp /content/frn-50k-baseline/latent_demand_recovery/exp --data_path /content/drive/MyDrive/FRN50K/raw/data/train.parquet --work_dir /content/task09_work/smoke --model TimesNet --missing_rate 0.3 --seed 1024 --batch_size 128 --pairs 200 --epochs 5 --verify_equivalence 

[task09] data_path=/content/drive/MyDrive/FRN50K/raw/data/train.parquet
[task09] train_rows=18000 pairs=200 dt_min=2024-03-28 dt_max=2024-06-25 stockout_points_6_21=57796 ram_gb=1.7
I0000 00:00:1794965782.412205    1413 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794965785.048970    1413 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
cuda:0
[task09] pypots_path=/content/frn-50k-baseline/latent_d

KeyboardInterrupt: 

## 6. Chạy đầy đủ (50.000 cặp, 5 epoch)
Lượt chính, ước tính 2–3 giờ trên T4. Giữ tab Colab mở, không để máy ngủ. Theo dõi các dòng `Epoch 00x` và `[task09] ... ram_gb=` trong log.

In [ ]:
FULL_LOG = OUT_DIR / 'results/full/timesnet_mnar.log'
full_cmd, full_rc = run_logged(['--epochs', str(EPOCHS), '--saving_path', str(CKPT_DIR)], FULL_LOG, WORK_ROOT / 'full')
full = parse_log(FULL_LOG)
RESUMED_FROM = None
print(json.dumps(full, indent=2))
def find_ckpt():
    found = [p for p in CKPT_DIR.rglob('*.pypots') if p.name == f'{MODEL}.pypots' and 'tensorboard' not in p.parts]
    return max(found, key=lambda p: p.stat().st_mtime) if found else None
print('Checkpoint trên Drive:', find_ckpt() or 'chưa có')

## 6b. CHỈ chạy ô này nếu mục 6 bị ngắt SAU KHI huấn luyện xong
Dùng khi log mục 6 đã có dòng `Saved the model to ...` nhưng chưa có `wape`/`wpe` (Colab bị ngắt hoặc tiến trình bị tắt, `mã thoát -9`, lúc dự đoán). Phải chạy bằng **đúng tài khoản Google đã chạy mục 6**, vì checkpoint nằm trên Drive của tài khoản đó. Ô này **không huấn luyện lại**: chạy lại mục 0 → 4, rồi chạy ô này để nạp checkpoint đã lưu trên Drive, dự đoán và tính WAPE/WPE (khoảng 15–30 phút). Nếu mục 6 đã chạy xong bình thường thì **bỏ qua** ô này.

In [9]:
def find_ckpt():
    found = [p for p in CKPT_DIR.rglob('*.pypots') if p.name == f'{MODEL}.pypots' and 'tensorboard' not in p.parts]
    return max(found, key=lambda p: p.stat().st_mtime) if found else None
RESUMED_FROM = find_ckpt()
assert RESUMED_FROM, 'Không thấy checkpoint TimesNet.pypots trên Drive – phải chạy lại mục 6 từ đầu.'
print(f'Kích thước checkpoint: {RESUMED_FROM.stat().st_size / 1024**2:.1f} MB')
print('Nạp checkpoint:', RESUMED_FROM)
FULL_LOG = OUT_DIR / 'results/full/timesnet_mnar_predict_from_ckpt.log'
full_cmd, full_rc = run_logged(['--epochs', str(EPOCHS), '--resume_ckpt', str(RESUMED_FROM)], FULL_LOG, WORK_ROOT / 'full')
full = parse_log(FULL_LOG)
print(json.dumps(full, indent=2))

Kích thước checkpoint: 5.9 MB
Nạp checkpoint: /content/drive/MyDrive/PTDLKD/task09_outputs/checkpoints/TimesNet/20261117_T014722/TimesNet.pypots
Lệnh: /usr/bin/python3 -u /content/run_task09.py --author_exp /content/frn-50k-baseline/latent_demand_recovery/exp --data_path /content/drive/MyDrive/FRN50K/raw/data/train.parquet --work_dir /content/task09_work/full --model TimesNet --missing_rate 0.3 --seed 1024 --batch_size 128 --epochs 5 --resume_ckpt /content/drive/MyDrive/PTDLKD/task09_outputs/checkpoints/TimesNet/20261117_T014722/TimesNet.pypots 

[task09] data_path=/content/drive/MyDrive/FRN50K/raw/data/train.parquet
[task09] train_rows=4500000 pairs=50000 dt_min=2024-03-28 dt_max=2024-06-25 stockout_points_6_21=14311536 ram_gb=2.1
I0000 00:00:1794965817.195200    1579 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild Ten

## 7. Lưu kết quả bàn giao

In [10]:
import math, csv

def finite(x):
    return x is not None and isinstance(x, float) and math.isfinite(x)

status = 'ok' if (full_rc == 0 and finite(full['wape']) and finite(full['wpe'])
                  and full['train_rows'] == 4_500_000) else 'failed'
row = {
    'model': MODEL, 'missing_rate': MISSING_RATE, 'seed': SEED, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
    'n_artificially_masked_hours': full['artificially_masked'] if full['artificially_masked'] is not None else '',
    'wape_pct': round(full['wape'] * 100, 4) if status == 'ok' else '',
    'wpe_pct': round(full['wpe'] * 100, 4) if status == 'ok' else '',
    'status': status,
    'log_path': str(FULL_LOG.relative_to(OUT_DIR)),
}
METRICS_CSV = OUT_DIR / 'results/mnar_metrics.csv'
with open(METRICS_CSV, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(row)); w.writeheader(); w.writerow(row)
print(pd.read_csv(METRICS_CSV).T)

if status == 'ok':
    print(f"\nĐối chiếu Bảng 2 (TimesNet): WAPE nhóm {row['wape_pct']:.2f}% vs bài báo {PAPER['wape_pct']:.2f}% "
          f"(chênh {row['wape_pct'] - PAPER['wape_pct']:+.2f} điểm %); "
          f"WPE nhóm {row['wpe_pct']:.2f}% vs bài báo {PAPER['wpe_pct']:.2f}% (chênh {row['wpe_pct'] - PAPER['wpe_pct']:+.2f} điểm %).")
else:
    print('>>> Lượt đầy đủ chưa có chỉ số hữu hạn: CSV ghi status=failed, để trống số (không điền 0).')

                                                                            0
model                                                                TimesNet
missing_rate                                                              0.3
seed                                                                     1024
epochs                                                                      5
batch_size                                                                128
n_artificially_masked_hours                                           9163425
wape_pct                                                              27.8141
wpe_pct                                                                2.0429
status                                                                     ok
log_path                     results/full/timesnet_mnar_predict_from_ckpt.log

Đối chiếu Bảng 2 (TimesNet): WAPE nhóm 27.81% vs bài báo 27.62% (chênh +0.19 điểm %); WPE nhóm 2.04% vs bài báo 1.43% (chênh +0.61 điểm %).


In [15]:
import importlib.metadata as im
def ver(p):
    try: return im.version(p)
    except Exception: return 'n/a'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
RUN_META = {
    'task': 'Task 9 - TimesNet MNAR',
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'author_repo': AUTHOR_REPO, 'author_commit': head,
    'data': DATA_INFO,
    'params': {'model': MODEL, 'missing_rate': MISSING_RATE, 'seed': SEED, 'epochs': EPOCHS,
               'batch_size': BATCH_SIZE, 'other_config': 'giữ nguyên CONFIG trong app.py (lr 0.001, d_model 64, n_layers 2, ...)'},
    'environment': {'python': platform.python_version(), 'torch': ver('torch'), 'numpy': ver('numpy'),
                    'pandas': ver('pandas'), 'pyarrow': ver('pyarrow'), 'pygrinder': ver('pygrinder'),
                    'tsdb': ver('tsdb'), 'gpu': gpu, 'ram_gb': round(psutil.virtual_memory().total / 1024**3, 1)},
    'smoke': {'pairs': SMOKE_PAIRS, 'epochs': SMOKE_EPOCHS, 'returncode': smoke_rc, **smoke},
    'full': {'returncode': full_rc, **full, 'status': status, 'resumed_from_checkpoint': str(RESUMED_FROM) if RESUMED_FROM else None},
    'commands': {'smoke': ' '.join(smoke_cmd), 'full': ' '.join(full_cmd)},
}
(OUT_DIR / 'results/run_metadata.json').write_text(json.dumps(RUN_META, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(RUN_META, indent=2, ensure_ascii=False))

{
  "task": "Task 9 - TimesNet MNAR",
  "run_utc": "2026-11-18T02:00:01.033792+00:00",
  "author_repo": "https://github.com/Dingdong-Inc/frn-50k-baseline.git",
  "author_commit": "8af694a09a5a427d1d50648d348c022565a781f2",
  "data": {
    "path": "/content/drive/MyDrive/FRN50K/raw/data/train.parquet",
    "sha256": "6706832db892bbae4969c19d87e07975d2543d2ba7d7d4756360654785de5a3d",
    "bytes": 106436287,
    "rows": 4500000,
    "columns": 19,
    "pairs": 50000,
    "all_pairs_90_days": true,
    "dt_min": "2024-03-28",
    "dt_max": "2024-06-25",
    "duplicate_keys": 0,
    "same_file_as_task6": false
  },
  "params": {
    "model": "TimesNet",
    "missing_rate": 0.3,
    "seed": 1024,
    "epochs": 5,
    "batch_size": 128,
    "other_config": "giữ nguyên CONFIG trong app.py (lr 0.001, d_model 64, n_layers 2, ...)"
  },
  "environment": {
    "python": "3.13.16",
    "torch": "2.14.1+cu130",
    "numpy": "2.4.6",
    "pandas": "2.2.3",
    "pyarrow": "23.0.1",
    "pygrinder": "0

In [12]:
SMOKE_LOG = OUT_DIR / 'results/smoke/timesnet_mnar_smoke.log'
smoke = parse_log(SMOKE_LOG)
smoke_rc = 0 if smoke['wape'] is not None else 1
smoke_cmd = SMOKE_LOG.read_text(encoding='utf-8').splitlines()[0].replace('# command: ', '').split()
print(json.dumps(smoke, indent=2))

{
  "wape": null,
  "wpe": null,
  "artificially_masked": 34390,
  "actual_mask_ratio": 0.149389,
  "stockout_points": 57796,
  "n_samples": 600,
  "train_rows": 18000,
  "elapsed_sec": null,
  "equivalence_ok": true,
  "eval_days": null,
  "peak_ram_gb": 2.9
}


In [14]:
def pct(x): return f'{x * 100:.2f}%' if finite(x) else 'không có'
info = f'''# Thông tin chạy Task 9 — TimesNet MNAR

**Cách tái lập:** mở `notebooks/09_timesnet_mnar.ipynb` trên Colab (GPU), sửa `DATA_PATH`, `OUT_DIR` ở mục 0 nếu cần, rồi chạy lần lượt các ô. Kết quả bàn giao do notebook xuất ra.

## Đầu vào

| Mục | Giá trị |
|---|---|
| File train | `{DATA_PATH.name}` ({DATA_INFO['bytes']:,} byte) |
| SHA-256 | `{DATA_SHA256}` |
| Trùng file Parquet Task 6 | {'Có' if DATA_INFO['same_file_as_task6'] else 'Không – bản tải trực tiếp từ Hugging Face (Dingdong-Inc/FreshRetailNet-50K, split train)'} |
| Số dòng / số cặp | {DATA_INFO['rows']:,} / {DATA_INFO['pairs']:,} (mỗi cặp 90 ngày) |
| Khoảng ngày | {DATA_INFO['dt_min']} – {DATA_INFO['dt_max']} |
| Eval | Không dùng |

## Mã và môi trường

- Repo nhóm: nhánh `feature/task09-timesnet-mnar`. Mã tác giả: `Dingdong-Inc/frn-50k-baseline` tại `{head}`, chạy `latent_demand_recovery/exp/app.py` với `pypots` đi kèm repo.
- Môi trường: Python {platform.python_version()}, torch {ver('torch')}, NumPy {ver('numpy')}, pandas {ver('pandas')}, pygrinder {ver('pygrinder')}, tsdb {ver('tsdb')}; GPU: {gpu}.
- Không cài lại torch 2.0.1/NumPy 1.23.5 theo `requirements.txt` của tác giả vì Colab hiện dùng Python mới; dùng bản có sẵn của Colab.

## Tham số thực tế

| Tham số | Giá trị | Ghi chú |
|---|---|---|
| model | {MODEL} | |
| missing_rate | {MISSING_RATE} | **Tham số mô phỏng theo ví dụ README của repo** (`python app.py --model TimesNet --missing_rate 0.3`); chưa xác minh là cấu hình chính xác của bài báo |
| Tỷ lệ che thực tế | {full['actual_mask_ratio']} | số điểm che thêm / số điểm giờ quan sát (không hết hàng) trong giờ 6..21 |
| Số điểm giờ che thêm | {full['artificially_masked']} | in bởi `load_data` của tác giả (`valid_idx.sum()`) |
| seed | {SEED} | giữ như CLI gốc (`set_seed(seed_value=1024)`) |
| epochs / batch size | {EPOCHS} / {BATCH_SIZE} | mặc định CONFIG của tác giả; các tham số khác giữ nguyên |

## Thay đổi so với mã tác giả

- Giữ nguyên: mô hình TimesNet và `pypots` đi kèm repo, `CONFIG` (trừ `saving_path`), `mimic_missing`, `set_seed`, `_imputation` (fit/predict), thứ tự gọi `set_seed` → `demand_recovery`.
- Viết lại `load_data` và `evaluation_mnar` để chạy được với 12,7 GB RAM (bản gốc bị Colab tắt tiến trình vì hết RAM ở bước tạo dữ liệu): đọc `train.parquet` local thay vì Hugging Face; sắp xếp theo `store_id, product_id, dt` như tác giả; tạo mảng đầu vào float32 bằng một lần cấp phát thay vì nhiều lần `np.concatenate` ở float64 (pypots vốn đổi sang float32 trước khi đưa vào mô hình); `evaluation_mnar` dùng lại mảng đã nạp thay vì nạp lại dữ liệu. Công thức và thứ tự gọi số ngẫu nhiên giữ nguyên. Hai hàm dùng chung một bộ dữ liệu → cùng tập, cùng thứ tự khóa.
- Kiểm chứng tương đương ở lượt chạy thử ({SMOKE_PAIRS} cặp, cùng seed): mảng đầu vào, mặt nạ điểm che thêm, `ts_origin` và WAPE/WPE của bản viết lại trùng bản gốc của tác giả → `equivalence_ok={smoke['equivalence_ok']}`.
- Truyền sẵn `norm_val` toàn số 1 dạng `(n, 1, 6)` cho pypots. Khi không có `norm_val`, pypots tự tạo `np.ones(X.shape)` ở float64 (3,5 GB, thêm bản float32 1,7 GB) mỗi lần tạo dataset, làm tiến trình bị tắt vì hết RAM ngay khi bắt đầu predict. Mô hình chỉ nhân đầu ra với `norm_val` theo broadcast nên kết quả không đổi (nhân với 1).
- Ghi thêm các dòng `[task09]` vào log (nguồn dữ liệu, số điểm che thêm, tỷ lệ che thực tế, RAM trước/sau huấn luyện và dự đoán, số ngày được đánh giá).
- Theo mã tác giả, WAPE/WPE tính trên tổng doanh số **theo ngày**, chỉ xét các ngày có điểm bị che thêm và không có hết hàng thực trong giờ 6..21.

## Kết quả

| Lượt | Quy mô | Epoch | WAPE | WPE | Trạng thái |
|---|---|---|---|---|---|
| Chạy thử (smoke) | {SMOKE_PAIRS} cặp | {SMOKE_EPOCHS} | {pct(smoke['wape'])} | {pct(smoke['wpe'])} | chỉ kiểm tra luồng chạy, không dùng làm kết quả |
| Đầy đủ (full) | 50.000 cặp | {EPOCHS} | {pct(full['wape'])} | {pct(full['wpe'])} | {status} |
| Bài báo (Bảng 2) | – | – | {PAPER['wape_pct']:.2f}% | {PAPER['wpe_pct']:.2f}% | số công bố, chỉ để đối chiếu |

{'Lượt đầy đủ được hoàn tất từ checkpoint `' + RESUMED_FROM.name + '` (huấn luyện ở `results/full/timesnet_mnar.log`, dự đoán + đánh giá ở `results/full/timesnet_mnar_predict_from_ckpt.log`) do Colab ngắt kết nối sau khi huấn luyện xong.' if RESUMED_FROM else 'Lượt đầy đủ chạy liền một mạch.'} Thời gian lượt cuối: {full['elapsed_sec']} giây; số ngày được đánh giá: {full['eval_days']}; RAM tiến trình cao nhất ghi nhận: {full['peak_ram_gb']} GB.

## File kết quả

- `results/mnar_metrics.csv`: 1 dòng TimesNet (WAPE/WPE theo %, giữ dấu WPE).
- `results/full/timesnet_mnar.log`: log lượt đầy đủ, chứa hai dòng `wape`, `wpe` (dạng tỷ lệ) dùng cho CSV.
- `results/smoke/timesnet_mnar_smoke.log`: log lượt chạy thử.
- `results/run_metadata.json`: môi trường, SHA-256, tham số, lệnh chạy.
- Checkpoint (`.pypots`) lưu trên Drive ở `checkpoints/`, file imputation `.npy` lưu tạm trên Colab (`{WORK_ROOT}`); cả hai không đưa lên GitHub.

## Lệnh đã chạy

```
{' '.join(full_cmd)}
```
'''
(OUT_DIR / 'run_info.md').write_text(info, encoding='utf-8')
print(info)
print('\nFile trong', OUT_DIR); [print(' -', p.relative_to(OUT_DIR)) for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]

# Thông tin chạy Task 9 — TimesNet MNAR

**Cách tái lập:** mở `notebooks/09_timesnet_mnar.ipynb` trên Colab (GPU), sửa `DATA_PATH`, `OUT_DIR` ở mục 0 nếu cần, rồi chạy lần lượt các ô. Kết quả bàn giao do notebook xuất ra.

## Đầu vào

| Mục | Giá trị |
|---|---|
| File train | `train.parquet` (106,436,287 byte) |
| SHA-256 | `6706832db892bbae4969c19d87e07975d2543d2ba7d7d4756360654785de5a3d` |
| Trùng file Parquet Task 6 | Không – bản tải trực tiếp từ Hugging Face (Dingdong-Inc/FreshRetailNet-50K, split train) |
| Số dòng / số cặp | 4,500,000 / 50,000 (mỗi cặp 90 ngày) |
| Khoảng ngày | 2024-03-28 – 2024-06-25 |
| Eval | Không dùng |

## Mã và môi trường

- Repo nhóm: nhánh `feature/task09-timesnet-mnar`. Mã tác giả: `Dingdong-Inc/frn-50k-baseline` tại `8af694a09a5a427d1d50648d348c022565a781f2`, chạy `latent_demand_recovery/exp/app.py` với `pypots` đi kèm repo.
- Môi trường: Python 3.13.16, torch 2.14.1+cu130, NumPy 2.4.6, pandas 2.2.3, pygrinder 0.4, tsdb 0.3.1; GPU: Tesla T4, 15360 Mi

[None, None, None, None, None, None, None]

## Nhận xét
- **Kết quả:** TimesNet với thiếu nhân tạo (`missing_rate = 0.3`, seed 1024, 5 epoch) đạt **WAPE 27,81%** và **WPE 2,04%** trên 783.078 ngày được đánh giá (ngày có điểm bị che thêm và không hết hàng thực trong giờ 6..21).
- **So với Bảng 2 (TimesNet: WAPE 27,62%, WPE 1,43%):** WAPE chênh +0,19 điểm phần trăm, WPE chênh +0,61 điểm phần trăm. WPE cùng dấu dương (dự báo hơi cao) như bài báo. Mức chênh nhỏ có thể do phiên bản thư viện/GPU khác tác giả (Python 3.13, torch 2.14 thay cho torch 2.0.1) và tính không tất định của huấn luyện trên GPU; cấu hình `missing_rate = 0.3` lấy theo ví dụ README, chưa xác minh là cấu hình chính xác của bài báo.
- **Tỷ lệ che thực tế:** 9.163.425 điểm giờ được che thêm, bằng 15,88% số điểm giờ quan sát được (không hết hàng) trong giờ 6..21 – khác con số 30% của tham số, vì `mimic_missing` dùng `missing_rate` làm xác suất theo từng đoạn ngày chứ không phải tỷ lệ che trực tiếp.
- **Lịch sử chạy:** huấn luyện đầy đủ chạy hai lần (16/11 và 17/11) cho cùng số điểm che thêm và cùng training loss từng epoch (0,0643 → 0,0619); cả hai lần tiến trình bị tắt vì hết RAM ngay khi bắt đầu dự đoán. Lần 17/11 đã lưu checkpoint lên Drive; ô 6b (18/11) nạp checkpoint đó, dự đoán và tính chỉ số sau khi sửa lỗi RAM bằng `norm_val` nhỏ. Ô mục 5 ở phiên 18/11 bị dừng tay sau khi kiểm chứng tương đương (`equivalence_ok=True`) vì không cần chạy lại phần huấn luyện thử.